<a href="https://colab.research.google.com/github/tamas2006/ai_chatbot/blob/main/notebook/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!ppip install unsloth transformers dataset trl accelerate peft bitsandbytes

/bin/bash: line 1: ppip: command not found


In [ ]:
!pip install unsloth transformers dataset trl accelerate peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
print("working")

working


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="bca_interview_dataset.json"
)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 30
    })
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

print("Model loaded successfully")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
def format_example(example):
  return{
      "text":f"instruction {example['instruction']}\nResponse: {example['response']}"
  }
dataset = dataset.map(format_example)
print(dataset["train"][0])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

{'instruction': 'Tell me about yourself', 'response': 'I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.', 'text': 'instruction Tell me about yourself\nResponse: I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.'}


In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=10,
    push_to_hub=False, # Explicitly set to False to avoid 'push_to_hub_token' error
    remove_unused_columns=False # Keep all columns for dataset processing
)

print("Training configuration ready")

Training configuration ready


In [ ]:
per_device_train_batch_size=1

In [ ]:
num_train_epochs=3

In [ ]:
logging_steps=1

In [ ]:
save_steps=10

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'text'],
        num_rows: 30
    })
})


In [ ]:
print(dataset["train"][0])

{'instruction': 'Tell me about yourself', 'response': 'I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.', 'text': 'instruction Tell me about yourself\nResponse: I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.'}


In [ ]:
print(type(model))

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [ ]:
print(type(model))

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [ ]:
print(tokenizer.pad_token)

</s>


In [ ]:
print(torch.cuda.is_available())
print(dataset["train"][0])
print(tokenizer.pad_token)
print(type(model))

True
{'instruction': 'Tell me about yourself', 'response': 'I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.', 'text': 'instruction Tell me about yourself\nResponse: I am a BCA graduate with interest in IT support, networking, and AI technologies. I enjoy learning new technologies and solving technical problems.'}
</s>
<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    args=training_args
)

trainer.train()

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

In [ ]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
)

trainer.train()

KeyError: 'push_to_hub_token'

In [ ]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
)

trainer.train()

KeyError: 'push_to_hub_token'

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Re-create TrainingArguments to include remove_unused_columns=False
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=10,
    remove_unused_columns=False # Explicitly keep all columns
)

# Tokenize the dataset
def tokenize_function(examples):
    # This assumes 'tokenizer' is accessible from previous cells.
    # It will generate 'input_ids' and 'attention_mask'.
    return tokenizer(examples["text"], truncation=True, padding="max_length")

# Map the tokenization function to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Initialize a data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator # Pass the data collator
)

trainer.train()

ValueError: No columns in the dataset match the model's forward method signature: (input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, labels, use_cache, logits_to_keep, kwargs, label_ids, label, labels). The following columns have been ignored: [instruction, text, response]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

Step,Training Loss
1,12.908222
2,5.642692
3,0.634029
4,0.574436
5,0.315748
6,0.441601
7,0.167616
8,0.497358
9,0.291816
10,0.400759


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=90, training_loss=0.3937030288080374, metrics={'train_runtime': 1667.8145, 'train_samples_per_second': 0.054, 'train_steps_per_second': 0.054, 'total_flos': 71505495982080.0, 'train_loss': 0.3937030288080374, 'epoch': 3.0})

In [ ]:
trainer.save_model("bca_interview_ai")
tokenizer.save_pretrained("bca_interview_ai")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bca_interview_ai/tokenizer_config.json',
 'bca_interview_ai/chat_template.jinja',
 'bca_interview_ai/tokenizer.json')

In [ ]:
prompt = "What is Python?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(tokenizer.decode(outputs[0]))

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s> What is Python?
Response: Python is a high-level programming language known for simplicity and readability. It is widely used in web development, automation, AI, and data science.</s>


In [ ]:
while True:
    question = input("Ask AI: ")

    if question.lower() == "exit":
        break

    ask_ai(question)

Ask AI: what is oops


NameError: name 'ask_ai' is not defined

In [ ]:
while True:
    question = input("Ask AI: ")

    if question.lower() == "exit":
        break

    ask_ai(question)

KeyboardInterrupt: Interrupted by user

In [ ]:
while True:
    question = input("Ask AI: ")

    if question.lower() == "exit":
        break

    ask_ai(question)

Ask AI: what is python


NameError: name 'ask_ai' is not defined

In [ ]:
def ask_ai(question):
    inputs = tokenizer(question, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(response)

In [ ]:
ask_ai("What is Python?")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is Python?
Response: Python is a high-level programming language known for simplicity and readability. It is widely used in web development, automation, AI, and data science.


In [ ]:
ask_ai("what is oops?")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


what is oops?
Response: Object-Oriented Programming is a programming approach based on objects and classes.


In [ ]:
trainer.save_model("bca_interview_ai")
tokenizer.save_pretrained("bca_interview_ai")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bca_interview_ai/tokenizer_config.json',
 'bca_interview_ai/chat_template.jinja',
 'bca_interview_ai/tokenizer.json')

In [ ]:
git init

SyntaxError: invalid syntax (2830201818.py, line 1)